# VERIONCO — Notebook 02: Genomic Pipeline (Full Dataset, 478 Cases)

**Purpose:** parse TCGA LUAD mutations, normalize variants, classify sample
type, match against CIViC evidence (variant-level and gene-level, separately),
retrieve resistance signals from PubMed and ClinicalTrials.gov, and produce
structured genomic evidence per patient.

**Design principles:**
- Reads `cohort_joined_wsi.csv` from Notebook 00 (full dataset: 478 cases / 468 genomic cases).
- Cross-checks patient overlap with Notebook 01's patch manifest.
- Variant-level CIViC matches are separated from gene-level matches. Gene-level
  matches are labelled as such and written to a file whose name warns consumers.
- Resistance association requires **functional** variants and a minimum
  gene-overlap count — it is not a raw gene-set intersection.
- PubMed and ClinicalTrials.gov parsing are checkpointed to JSONL.
- Per-patient evidence is written to a directory to avoid a giant JSON.
- CIViC's missing evidence_direction / clinical_significance / evidence_level
  are documented as limitations, not faked.

**Leakage contract:** every genomic evidence record carries `patient_id`;
Notebook 03+ join clinical features on the same key.

## INPUTS

| Source | Path | Used for |
|--------|------|----------|
| Notebook 00 outputs | `/kaggle/input/.../VERIONCO_outputs/00/` | `cohort_joined_wsi.csv`, `patient_master.csv`, `modality_completeness.csv` |
| Notebook 01 outputs | `/kaggle/input/.../pathology-pipeline-01/` (or new NB01 path) | `patch_manifest.csv` for patient-overlap check |
| CIViC evidence | `civic_evidence.tsv` | Variant / gene evidence matching |
| TCGA mutations | `luad_tcga_pan_can_atlas_2018/data_mutations.txt` | Patient somatic variants |
| PubMed MEDLINE | `pubmed.txt` | Resistance evidence |
| ClinicalTrials.gov | `clinicaltrials.json/*.json` | Resistance evidence |

## CONFIGURATION

Edit only the `PATHS` block to match your Kaggle attachment names.

In [1]:
# ============================================================
# CONFIGURATION  ← edit only this cell
# ============================================================
from pathlib import Path
import os, gc, sys, json, hashlib, warnings, time, traceback
from datetime import datetime, timezone
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Detect environment
IS_KAGGLE = Path("/kaggle").exists()

def find_dir(candidates, fallback):
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    return Path(fallback)

OUTPUT_ROOT = Path("/kaggle/working/VERIONCO_outputs") if IS_KAGGLE else Path("./VERIONCO_outputs")
NB_OUTPUT       = OUTPUT_ROOT / "02"
NB_OUTPUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints" / "02"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

NB00_ROOT = find_dir([
    "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00",
    "/kaggle/input/verionco-outputs-00/00",
    "/kaggle/input/verionco-outputs-00",
    "/kaggle/input/verionco-outputs-local/00",
    "/kaggle/working/VERIONCO_outputs/00",
    "./VERIONCO_outputs/00"
], fallback="./VERIONCO_outputs/00" if not IS_KAGGLE else "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00")

NB01_ROOT = find_dir([
    "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-01",
    "/kaggle/input/verionco-outputs-01",
    "/kaggle/input/verionco-outputs-local/01",
    "/kaggle/working/VERIONCO_outputs/01",
    "./VERIONCO_outputs/01"
], fallback="./VERIONCO_outputs/01" if not IS_KAGGLE else "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-01")

RETRIEVAL = find_dir([
    "/kaggle/input/datasets/ayushsrivastava2000/raw-retrival",
    "/kaggle/input/raw-retrival",
    "/kaggle/input/datasets/ayushsrivastava2000/retrieval",
    "/kaggle/input/retrieval",
    "./data/retrieval"
], fallback="./data/retrieval" if not IS_KAGGLE else "/kaggle/input/datasets/ayushsrivastava2000/raw-retrival")

TCGA_ROOT = find_dir([
    "/kaggle/input/datasets/abhigyansinha09/tcgadataset",
    "/kaggle/input/tcgadataset",
    "/kaggle/input/datasets/abhigyansinha09/tcga",
    "/kaggle/input/tcga",
    "./data/tcga"
], fallback="./data/tcga" if not IS_KAGGLE else "/kaggle/input/datasets/abhigyansinha09/tcgadataset")

PATHS = {
    "nb00":      NB00_ROOT,
    "nb01":      NB01_ROOT,
    "retrieval": RETRIEVAL,
    "tcga":      TCGA_ROOT,
}

# ---- Variant filtering --------------------------------------
MIN_VARIANT_DEPTH = 10
MIN_TUMOR_VAF     = 0.05
INCLUDE_SILENT    = False
ESM2_ENABLED      = False

# Variant classifications we treat as potentially functional
FUNCTIONAL_CLASSES = {
    "Missense_Mutation", "Nonsense_Mutation",
    "Frame_Shift_Del", "Frame_Shift_Ins",
    "Splice_Site", "In_Frame_Del", "In_Frame_Ins",
    "Nonstop_Mutation", "Translation_Start_Site",
}

# ---- Resistance association rule ----------------------------
MIN_RES_GENE_OVERLAP = 2
HYPERMUTATION_FACTOR = 5.0

EXPECTED_MIN_CASES = 10
EXPECTED_MAX_CASES = 550

PROJECT = "VERIONCO"
NOTEBOOK = "02_genomic_pipeline"

for k, v in PATHS.items():
    print(f"{k:12s} | {v} | exists={v.exists()}")

assert PATHS["nb00"].exists(), f"Notebook 00 root missing: {PATHS['nb00']}"
assert PATHS["retrieval"].exists(), f"Retrieval root missing: {PATHS['retrieval']}"
assert PATHS["tcga"].exists(), f"TCGA root missing: {PATHS['tcga']}"
if not PATHS["nb01"].exists():
    warnings.warn(f"Notebook 01 root missing: {PATHS['nb01']} — overlap check will be skipped.")

print(f"\nOutput:      {NB_OUTPUT}")
print(f"Checkpoints: {CHECKPOINT_ROOT}")


nb00         | VERIONCO_outputs\00 | exists=True
nb01         | VERIONCO_outputs\01 | exists=False
retrieval    | data\retrieval | exists=True
tcga         | data\tcga | exists=True

Output:      VERIONCO_outputs\02
Checkpoints: VERIONCO_outputs\checkpoints\02


<string>:103: UserWarning: Notebook 01 root missing: VERIONCO_outputs\01 — overlap check will be skipped.


## Environment check

In [2]:
import importlib.util
packages = ["pandas","numpy","matplotlib","seaborn","sklearn"]
rows = [{"package": p, "available": importlib.util.find_spec(p) is not None} for p in packages]
display(pd.DataFrame(rows))

runtime = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
}
print(json.dumps(runtime, indent=2))

,package,available
0,pandas,True
1,numpy,True
2,matplotlib,True
3,seaborn,True
4,sklearn,True


{
  "timestamp_utc": "2026-10-19T13:19:21.163689+00:00",
  "python": "3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]"
}


## LOAD — cohort from Notebook 00, cross-check with Notebook 01/

In [3]:
# ============================================================
# LOAD — cohort from Notebook 00 + NB01 overlap check
# ============================================================
t0 = time.time()

cohort_path = PATHS["nb00"] / "cohort_joined_wsi.csv"
assert cohort_path.exists(), (
    f"Missing {cohort_path}. Run the corrected Notebook 00 first."
)

cohort = pd.read_csv(cohort_path)
print(f"cohort_joined_wsi.csv: {cohort.shape}")
print(f"columns: {list(cohort.columns)}")

# Normalize filename column name
if "filename" in cohort.columns and "file_name" not in cohort.columns:
    cohort = cohort.rename(columns={"filename": "file_name"})

n_cases    = len(cohort)
n_patients = cohort["patient_id"].nunique()
print(f"\ncohort: {n_cases} cases | {n_patients} unique patients")

assert cohort["patient_id"].isna().sum() == 0, "Missing patient_id in cohort."

patient_set = set(cohort["patient_id"].dropna().unique())

# Cross-check against Notebook 01 patch manifest (if available)
overlap_path = NB_OUTPUT / "patient_modality_overlap.csv"
if PATHS["nb01"].exists():
    patch_paths = list(PATHS["nb01"].rglob("patch_manifest.csv"))
    if patch_paths:
        patch_manifest = pd.read_csv(patch_paths[0], usecols=["patient_id"])
        path_patients = set(patch_manifest["patient_id"].dropna().unique())
        overlap = patient_set & path_patients
        print(f"\npatient overlap:")
        print(f"  genomic patients:         {len(patient_set)}")
        print(f"  pathology patients:       {len(path_patients)}")
        print(f"  intersection:             {len(overlap)}")
        only_genomic = sorted(patient_set - path_patients)
        only_path    = sorted(path_patients - patient_set)
        if only_genomic:
            print(f"  genomic-only patients:    {only_genomic[:10]}{'...' if len(only_genomic)>10 else ''}")
        if only_path:
            print(f"  pathology-only patients:  {only_path[:10]}{'...' if len(only_path)>10 else ''}")

        union = sorted(patient_set | path_patients)
        pd.DataFrame({
            "patient_id":   union,
            "in_genomic":   [p in patient_set   for p in union],
            "in_pathology": [p in path_patients for p in union],
        }).to_csv(overlap_path, index=False)
        print(f"  saved: {overlap_path}")
    else:
        print("No patch_manifest.csv found under NB01 root.")
else:
    print("NB01 root not available; skipping overlap check.")

print(f"\n[✓] cohort loaded in {time.time()-t0:.1f}s")

cohort_joined_wsi.csv: (478, 9)
columns: ['file_id', 'filename', 'patient_id', 'found', 'n_candidates', 'roots', 'paths', 'md5_ok', 'size_ok']

cohort: 478 cases | 478 unique patients
NB01 root not available; skipping overlap check.

[✓] cohort loaded in 0.0s


## PROCESSING — TCGA mutations

In [4]:
# ============================================================
# PROCESSING — load mutations, map to patients, classify sample type
# ============================================================
def resolve_tcga_file(file_name):
    for p in PATHS["tcga"].rglob(file_name):
        if p.is_file():
            return p
    raise FileNotFoundError(f"TCGA file not found: {file_name}")

MUTATION_FILE = resolve_tcga_file("data_mutations.txt")
print(f"mutation file: {MUTATION_FILE}")

mutation_df = pd.read_csv(MUTATION_FILE, sep="\t", low_memory=False, comment="#")
print(f"raw mutations: {len(mutation_df):,} rows, {len(mutation_df.columns)} cols")

# Barcode column
barcode_col = None
for cand in ["Tumor_Sample_Barcode", "SAMPLE_ID", "Sample_Id", "sample_id"]:
    if cand in mutation_df.columns:
        barcode_col = cand
        break
assert barcode_col is not None, f"No sample barcode column. Columns: {list(mutation_df.columns)}"
print(f"barcode column: {barcode_col}")

def barcode_to_patient(barcode):
    if pd.isna(barcode):
        return None
    parts = str(barcode).split("-")
    return "-".join(parts[:3]) if len(parts) >= 3 else None

def classify_sample_type(barcode):
    """TCGA sample type codes: 01 primary tumor, 02 recurrent, 06 metastatic, 11 normal."""
    if pd.isna(barcode):
        return "unknown"
    parts = str(barcode).split("-")
    if len(parts) < 4:
        return "unknown"
    code = parts[3][:2]
    return {
        "01": "primary_tumor",
        "02": "recurrent_tumor",
        "06": "metastatic",
        "11": "solid_tissue_normal",
    }.get(code, "other")

mutation_df["patient_id"]  = mutation_df[barcode_col].apply(barcode_to_patient)
mutation_df["sample_id"]   = mutation_df[barcode_col]
mutation_df["sample_type"] = mutation_df[barcode_col].apply(classify_sample_type)

# Restrict to cohort patients
cohort_mutations = mutation_df[mutation_df["patient_id"].isin(patient_set)].copy()
print(f"cohort mutations: {len(cohort_mutations):,} across "
      f"{cohort_mutations['patient_id'].nunique()} patients")
print(f"\nsample_type distribution (cohort):")
print(cohort_mutations["sample_type"].value_counts().to_string())

mutation file: data\tcga\luad_tcga_pan_can_atlas_2018\data_mutations.txt
raw mutations: 225,653 rows, 114 cols
barcode column: Tumor_Sample_Barcode
cohort mutations: 190,887 across 468 patients

sample_type distribution (cohort):
sample_type
primary_tumor    190887


In [5]:
# ============================================================
# PROCESSING — filters: silent, depth, VAF
# ============================================================
# Silent filter
if not INCLUDE_SILENT and "Variant_Classification" in cohort_mutations.columns:
    before = len(cohort_mutations)
    cohort_mutations = cohort_mutations[
        cohort_mutations["Variant_Classification"] != "Silent"
    ].copy()
    print(f"removed {before - len(cohort_mutations)} silent")

# Depth filter
if "t_depth" in cohort_mutations.columns:
    before = len(cohort_mutations)
    cohort_mutations = cohort_mutations[
        cohort_mutations["t_depth"].fillna(0) >= MIN_VARIANT_DEPTH
    ].copy()
    print(f"removed {before - len(cohort_mutations)} low-depth (<{MIN_VARIANT_DEPTH})")

# VAF
if "t_alt_count" in cohort_mutations.columns and "t_depth" in cohort_mutations.columns:
    cohort_mutations["tumor_vaf"] = (
        cohort_mutations["t_alt_count"] /
        cohort_mutations["t_depth"].replace(0, np.nan)
    )
    before = len(cohort_mutations)
    cohort_mutations = cohort_mutations[
        cohort_mutations["tumor_vaf"].fillna(0) >= MIN_TUMOR_VAF
    ].copy()
    print(f"removed {before - len(cohort_mutations)} low-VAF (<{MIN_TUMOR_VAF})")
else:
    cohort_mutations["tumor_vaf"] = np.nan

print(f"\nfinal cohort mutations: {len(cohort_mutations):,}")

# Hypermutation flag
per_patient_counts = cohort_mutations.groupby("patient_id").size()
median_count = float(per_patient_counts.median())
hypermutated = set(
    per_patient_counts[per_patient_counts > HYPERMUTATION_FACTOR * median_count].index
)
print(f"per-patient median mutations: {median_count:.0f}")
print(f"hypermutated patients (>{HYPERMUTATION_FACTOR}x median): {sorted(hypermutated)}")

removed 38846 silent
removed 613 low-depth (<10)
removed 2313 low-VAF (<0.05)

final cohort mutations: 149,115
per-patient median mutations: 222
hypermutated patients (>5.0x median): ['TCGA-05-4382', 'TCGA-05-4396', 'TCGA-44-7670', 'TCGA-49-AARE', 'TCGA-53-7626', 'TCGA-55-7907', 'TCGA-55-7994', 'TCGA-55-8089', 'TCGA-55-8506', 'TCGA-55-A490', 'TCGA-64-5781', 'TCGA-69-7979', 'TCGA-78-7155', 'TCGA-78-8662', 'TCGA-86-8073', 'TCGA-86-8358', 'TCGA-86-A4JF', 'TCGA-95-7039', 'TCGA-95-7043', 'TCGA-L9-A7SV', 'TCGA-MN-A4N4', 'TCGA-NJ-A4YQ']


In [6]:
# ============================================================
# PROCESSING — variant normalization + gene-level aggregation
# ============================================================
def normalize_variant(row):
    gene    = str(row.get("Hugo_Symbol", "")).strip()
    hgvsp   = str(row.get("HGVSp_Short", "")).strip()
    var_cls = str(row.get("Variant_Classification", "")).strip()
    chrom   = str(row.get("Chromosome", "")).strip()
    start   = str(row.get("Start_Position", "")).strip()
    ref     = str(row.get("Reference_Allele", "")).strip()
    alt     = str(row.get("Tumor_Seq_Allele2", "")).strip()
    if hgvsp and hgvsp.lower() != "nan":
        return f"{gene}:{hgvsp}"
    if var_cls and var_cls.lower() != "nan":
        return f"{gene}:{var_cls}"
    return f"{gene}:{chrom}:{start}:{ref}>{alt}"

cohort_mutations["variant_canonical"] = cohort_mutations.apply(normalize_variant, axis=1)

gene_summary = (
    cohort_mutations.groupby(["patient_id", "Hugo_Symbol"])
    .agg(
        n_variants         = ("variant_canonical", "size"),
        variant_classes    = ("Variant_Classification", lambda x: sorted(set(x))),
        canonical_variants = ("variant_canonical", lambda x: sorted(set(x))),
    )
    .reset_index()
)
print(f"gene-level records: {len(gene_summary)}")

# Functional-gene sets per patient (used later for resistance association)
functional_genes_per_patient = (
    cohort_mutations[cohort_mutations["Variant_Classification"].isin(FUNCTIONAL_CLASSES)]
    .groupby("patient_id")["Hugo_Symbol"]
    .apply(lambda s: {str(g).upper() for g in s})
    .to_dict()
)
print(f"functional gene sets: {len(functional_genes_per_patient)} patients")

gene-level records: 138745
functional gene sets: 467 patients


## PROCESSING — CIViC matching (variant-level and gene-level, separated)

In [7]:
# ============================================================
# PROCESSING — CIViC load, granularity report, variant/gene parsing
# ============================================================
import re
from collections import defaultdict
import numpy as np
import pandas as pd

civic_path = PATHS["retrieval"] / "civic_evidence.tsv"
civic_df = pd.read_csv(civic_path, sep="\t", low_memory=False)
print(f"CIViC rows: {len(civic_df):,}, cols: {list(civic_df.columns)}")

# Normalize CIViC schema fields across nightly/API formats
if "molecular_profile" in civic_df.columns:
    civic_df["feature_name"] = civic_df["molecular_profile"].astype(str).str.split().str[0]
    civic_df["name"] = civic_df["molecular_profile"]
    civic_df["id"] = civic_df.get("evidence_id", civic_df.index)
    civic_df["category"] = civic_df.get("evidence_type", "Somatic")
    civic_df["clinical_significance"] = civic_df.get("significance", "Unknown")
    civic_df["diseases"] = civic_df.get("disease", "")

# Gene column
civic_gene_col = None
for cand in ["feature_name", "gene", "Gene", "gene_name", "molecular_profile"]:
    if cand in civic_df.columns:
        civic_gene_col = cand
        break
assert civic_gene_col is not None, "No gene column found in CIViC"

CIVIC_EVIDENCE_FIELDS = {"evidence_direction", "clinical_significance",
                          "evidence_level", "evidence_statement", "description"}
civic_present = CIVIC_EVIDENCE_FIELDS & set(civic_df.columns)
civic_missing = CIVIC_EVIDENCE_FIELDS - set(civic_df.columns)
civic_granularity = ("evidence_item_level" if not civic_missing
                     else "feature_level")
print(f"civic granularity: {civic_granularity}")
print(f"missing evidence fields: {sorted(civic_missing)}")

# Optionally drop zero-evidence features
if "evidence_item_count" in civic_df.columns:
    before = len(civic_df)
    civic_df["evidence_item_count"] = pd.to_numeric(
        civic_df["evidence_item_count"], errors="coerce").fillna(0)
    civic_df = civic_df[civic_df["evidence_item_count"] > 0].copy()
    print(f"dropped {before - len(civic_df)} CIViC features with evidence_item_count == 0")

# Parse variants from CIViC name field
def parse_variant_from_name(name, gene):
    if pd.isna(name) or gene is None:
        return None
    s = str(name).strip()
    if not s:
        return None
    su = s.upper()
    if su == gene:
        return None
    prefix = gene + " "
    if su.startswith(prefix):
        return s[len(prefix):].strip().upper()
    if " " not in s and re.search(r"\d", s):
        return su
    parts = s.split(None, 1)
    if len(parts) == 2 and parts[0].upper() == gene:
        return parts[1].strip().upper()
    return None

civic_df["gene_upper"]     = civic_df[civic_gene_col].astype(str).str.upper().str.strip()
civic_df["parsed_gene"]    = civic_df["gene_upper"]
civic_df["parsed_variant"] = [
    parse_variant_from_name(n, g)
    for n, g in zip(civic_df.get("name", pd.Series([None]*len(civic_df))),
                    civic_df["parsed_gene"])
]

civic_by_gene    = defaultdict(list)
civic_by_variant = defaultdict(list)
for _, hit in civic_df.iterrows():
    g = hit["parsed_gene"]
    v = hit["parsed_variant"]
    civic_by_gene[g].append(hit)
    if v:
        civic_by_variant[(g, v)].append(hit)

cohort_genes = set(cohort_mutations["Hugo_Symbol"].astype(str).str.upper())
matched_genes = cohort_genes & set(civic_by_gene.keys())
print(f"cohort genes: {len(cohort_genes)} | matched in CIViC: {len(matched_genes)}")


CIViC rows: 4,952, cols: ['molecular_profile', 'molecular_profile_id', 'disease', 'doid', 'phenotypes', 'therapies', 'therapy_interaction_type', 'evidence_type', 'evidence_direction', 'evidence_level', 'significance', 'evidence_statement', 'citation_id', 'source_type', 'asco_abstract_id', 'citation', 'nct_ids', 'rating', 'evidence_status', 'evidence_id', 'variant_origin', 'last_review_date', 'evidence_civic_url', 'molecular_profile_civic_url', 'is_flagged']
civic granularity: feature_level
missing evidence fields: ['description']
cohort genes: 18311 | matched in CIViC: 311


In [8]:
# ============================================================
# PROCESSING — CIViC matching, with variant / gene separation
# ============================================================
VARIANT_CLASS_TOKENS = {
    "MISSENSE_MUTATION", "NONSENSE_MUTATION", "FRAME_SHIFT_INS",
    "FRAME_SHIFT_DEL", "SPLICE_SITE", "IN_FRAME_DEL", "IN_FRAME_INS",
    "NONSTOP_MUTATION", "TRANSLATION_START_SITE",
}

def extract_cohort_variant(variant_canonical):
    if pd.isna(variant_canonical):
        return None
    parts = str(variant_canonical).split(":")
    if len(parts) < 2:
        return None
    tok = parts[1].strip().upper()
    tok = re.sub(r"^P\.", "", tok)
    if tok in VARIANT_CLASS_TOKENS:
        return None
    return tok

rows = []
for _, r in cohort_mutations.iterrows():
    g = str(r["Hugo_Symbol"]).upper()
    v = extract_cohort_variant(r["variant_canonical"])

    exact_hits = civic_by_variant.get((g, v), []) if v else []
    fallback   = civic_by_gene.get(g, [])

    if exact_hits:
        hits, level = exact_hits, "variant"
    elif fallback:
        hits, level = fallback, "gene"
    else:
        continue

    for h in hits:
        rows.append({
            "patient_id":        r["patient_id"],
            "sample_id":         r["sample_id"],
            "sample_type":       r.get("sample_type"),
            "gene":              r["Hugo_Symbol"],
            "variant_canonical": r["variant_canonical"],
            "variant_class":     r.get("Variant_Classification"),
            "match_level":       level,
            "civic_id":          h.get("id"),
            "civic_name":        h.get("name"),
            "civic_category":    h.get("category"),
            "civic_variant_types": h.get("variant_types"),
            "civic_therapies":   h.get("therapies"),
            "civic_diseases":    h.get("diseases"),
            "civic_evidence_item_count": h.get("evidence_item_count"),
        })

civic_matches_df = pd.DataFrame(rows)

# Separate variant-level and gene-level matches
civic_variant_df = civic_matches_df[civic_matches_df["match_level"] == "variant"].copy() \
                   if len(civic_matches_df) else pd.DataFrame()
civic_gene_df    = civic_matches_df[civic_matches_df["match_level"] == "gene"].copy() \
                   if len(civic_matches_df) else pd.DataFrame()

n_var = len(civic_variant_df)
n_gen = len(civic_gene_df)
n_tot = len(civic_matches_df)
variant_fraction = n_var / max(1, n_tot)

print(f"CIViC matches total:  {n_tot}")
print(f"  variant-level:      {n_var}  ({variant_fraction*100:.1f}%)")
print(f"  gene-level:         {n_gen}  ({(1-variant_fraction)*100:.1f}%)")
print(f"\n>>> CIViC match quality: {'LOW' if variant_fraction < 0.5 else 'ACCEPTABLE'}")
if variant_fraction < 0.5:
    print(">>> Gene-level matches dominate. Downstream notebooks MUST NOT treat")
    print(">>> gene-level matches as clinical evidence.")

CIViC matches total:  93476
  variant-level:      4169  (4.5%)
  gene-level:         89307  (95.5%)

>>> CIViC match quality: LOW
>>> Gene-level matches dominate. Downstream notebooks MUST NOT treat
>>> gene-level matches as clinical evidence.


## PROCESSING — resistance evidence (PubMed + ClinicalTrials.gov), checkpointed

In [9]:
# ============================================================
# PROCESSING — PubMed MEDLINE parser + resistance extraction
# ============================================================
def parse_medline(path):
    records = []
    cur = {}
    key = None
    buf = []

    def flush():
        nonlocal key, buf
        if key and buf:
            cur.setdefault(key, []).append(" ".join(buf).strip())
        key, buf = None, []

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for raw in f:
            line = raw.rstrip("\n")
            if not line.strip():
                continue
            if line.startswith("PMID- "):
                flush()
                if cur:
                    records.append(cur)
                    cur = {}
                key, buf = "PMID", [line[6:].strip()]
            elif line.startswith("      ") and key:
                buf.append(line.strip())
            else:
                m = re.match(r"^([A-Z]{2,4})\s*- (.*)$", line)
                if m:
                    flush()
                    key, buf = m.group(1), [m.group(2).strip()]
    flush()
    if cur:
        records.append(cur)
    return records

RESISTANCE_RE = re.compile(
    r"\b(resistan(?:ce|t)|refractory|acquired resistance|treatment failure|"
    r"non[- ]?response|bypass(?:ing)? (?:pathway|mechanism)|"
    r"secondary mutation)\b",
    re.IGNORECASE,
)
TOKEN_SPLIT_RE = re.compile(r"[^A-Za-z0-9_\-]+")

def find_cohort_genes(text, gene_set_upper):
    tokens = set(TOKEN_SPLIT_RE.split(text.upper()))
    return sorted(tokens & gene_set_upper)

cohort_genes_upper = {g.upper() for g in cohort_genes}

pubmed_path = PATHS["retrieval"] / "pubmed.txt"
pubmed_cache = CHECKPOINT_ROOT / "pubmed_resistance.jsonl"

if pubmed_cache.exists():
    print(f"Loading cached PubMed resistance from {pubmed_cache}")
    with open(pubmed_cache, "r", encoding="utf-8") as f:
        resistance_pubmed = [json.loads(l) for l in f if l.strip()]
    print(f"cached PubMed resistance records: {len(resistance_pubmed)}")
else:
    print(f"Parsing PubMed: {pubmed_path}")
    pubmed_records = parse_medline(pubmed_path) if pubmed_path.exists() else []
    print(f"parsed {len(pubmed_records):,} MEDLINE records")

    resistance_pubmed = []
    n_res_terms = 0
    for rec in pubmed_records:
        title = " ".join(rec.get("TI", []))
        abst  = " ".join(rec.get("AB", []))
        mesh  = " ".join(rec.get("MH", []))
        text  = f"{title} {abst} {mesh}"
        if not RESISTANCE_RE.search(text):
            continue
        n_res_terms += 1
        genes = find_cohort_genes(text, cohort_genes_upper)
        if not genes:
            continue
        resistance_pubmed.append({
            "source":              "pubmed",
            "source_id":           rec.get("PMID", [None])[0],
            "title":               title[:400],
            "genes_mentioned":     genes,
            "has_resistance_term": True,
        })
    print(f"PubMed resistance records: {len(resistance_pubmed)} "
          f"(scanned {n_res_terms:,} resistance-containing abstracts)")

    # Checkpoint
    with open(pubmed_cache, "w", encoding="utf-8") as f:
        for rec in resistance_pubmed:
            f.write(json.dumps(rec, default=str) + "\n")
    print(f"cached to {pubmed_cache}")

Loading cached PubMed resistance from VERIONCO_outputs\checkpoints\02\pubmed_resistance.jsonl
cached PubMed resistance records: 21


In [10]:
# ============================================================
# PROCESSING — ClinicalTrials.gov resistance extraction (checkpointed)
# ============================================================
ct_dir = PATHS["retrieval"] / "clinicaltrials.json"
ct_cache = CHECKPOINT_ROOT / "clinicaltrials_resistance.jsonl"

if ct_cache.exists():
    print(f"Loading cached ClinicalTrials resistance from {ct_cache}")
    with open(ct_cache, "r", encoding="utf-8") as f:
        resistance_ct = [json.loads(l) for l in f if l.strip()]
    print(f"cached CT.gov resistance records: {len(resistance_ct)}")
else:
    resistance_ct = []
    if ct_dir.exists():
        ct_files = sorted(ct_dir.glob("*.json"))
        print(f"scanning {len(ct_files):,} ClinicalTrials.gov JSON files")
        for i, jf in enumerate(ct_files):
            if i and i % 500 == 0:
                print(f"  ... {i}/{len(ct_files)}")
            try:
                with open(jf, "r", encoding="utf-8", errors="replace") as f:
                    obj = json.load(f)
            except Exception:
                continue

            ps   = obj.get("protocolSection", {})
            ident = ps.get("identificationModule", {})
            descr = ps.get("descriptionModule", {})
            elig  = ps.get("eligibilityModule", {})
            arms  = ps.get("armsInterventionsModule", {})
            cond  = ps.get("conditionsModule", {})

            interventions = arms.get("interventions", []) or []
            interv_txt = " ".join(
                f"{i.get('name','')} {i.get('description','')}"
                for i in interventions if isinstance(i, dict)
            )
            conds = cond.get("conditions", []) or []

            blob = " ".join([
                str(ident.get("briefTitle", "")),
                str(ident.get("officialTitle", "")),
                str(descr.get("briefSummary", "")),
                str(descr.get("detailedDescription", "")),
                str(elig.get("eligibilityCriteria", "")),
                " ".join(str(c) for c in conds),
                interv_txt,
            ])
            if not blob.strip():
                continue
            if not RESISTANCE_RE.search(blob):
                continue
            genes = find_cohort_genes(blob, cohort_genes_upper)
            if not genes:
                continue
            resistance_ct.append({
                "source":              "clinicaltrials_gov",
                "source_id":           ident.get("nctId"),
                "title":               (ident.get("briefTitle") or "")[:400],
                "genes_mentioned":     genes,
                "has_resistance_term": True,
            })
        print(f"ClinicalTrials.gov resistance records: {len(resistance_ct)}")

        with open(ct_cache, "w", encoding="utf-8") as f:
            for rec in resistance_ct:
                f.write(json.dumps(rec, default=str) + "\n")
        print(f"cached to {ct_cache}")
    else:
        print(f"ClinicalTrials.gov directory not found: {ct_dir}")

resistance_df = pd.DataFrame(resistance_pubmed + resistance_ct)
print(f"\ntotal resistance evidence records: {len(resistance_df)}")

Loading cached ClinicalTrials resistance from VERIONCO_outputs\checkpoints\02\clinicaltrials_resistance.jsonl
cached CT.gov resistance records: 0

total resistance evidence records: 21


## EVALUATION — QC report + dashboard

In [11]:
# ============================================================
# EVALUATION — QC report + dashboard
# ============================================================
patient_mutation_counts = cohort_mutations.groupby("patient_id").size().to_dict()
gene_freq  = cohort_mutations.groupby("Hugo_Symbol").size().sort_values(ascending=False)
class_dist = cohort_mutations["Variant_Classification"].value_counts().to_dict()
per_patient_civic = civic_matches_df.groupby("patient_id").size().to_dict() if len(civic_matches_df) else {}

qc = {
    "notebook":                     NOTEBOOK,
    "timestamp_utc":                datetime.now(timezone.utc).isoformat(),
    "cohort_cases":                 int(n_cases),
    "cohort_patients":              int(n_patients),
    "total_mutations_raw":          int(len(mutation_df)),
    "cohort_mutations_filtered":    int(len(cohort_mutations)),
    "patient_mutation_counts":      patient_mutation_counts,
    "unique_genes":                 int(cohort_mutations["Hugo_Symbol"].nunique()),
    "variant_class_distribution":   class_dist,
    "hypermutated_patients":        sorted(hypermutated),
    "hypermutation_factor":         HYPERMUTATION_FACTOR,
    "civic_granularity":            civic_granularity,
    "civic_missing_fields":         sorted(civic_missing),
    "civic_matches_total":          int(n_tot),
    "civic_matches_variant_level":  int(n_var),
    "civic_matches_gene_level":     int(n_gen),
    "civic_variant_level_fraction": float(variant_fraction),
    "civic_match_quality":          ("low" if variant_fraction < 0.5 else "acceptable"),
    "civic_matches_per_patient":    per_patient_civic,
    "resistance_total":             int(len(resistance_df)),
    "resistance_pubmed":            int((resistance_df["source"] == "pubmed").sum()) if len(resistance_df) else 0,
    "resistance_clinicaltrials":    int((resistance_df["source"] == "clinicaltrials_gov").sum()) if len(resistance_df) else 0,
    "resistance_min_gene_overlap":  MIN_RES_GENE_OVERLAP,
    "min_depth_filter":             MIN_VARIANT_DEPTH,
    "min_vaf_filter":               MIN_TUMOR_VAF,
    "include_silent":               INCLUDE_SILENT,
    "esm2_enabled":                 ESM2_ENABLED,
    "patient_leakage_note":         "genomic evidence joins on patient_id",
}

# Sanity warnings
if variant_fraction < 0.5:
    qc["civic_warning"] = (
        f"Only {variant_fraction*100:.1f}% of CIViC matches are variant-level. "
        "Gene-level matches are NOT clinical evidence."
    )

if len(resistance_df):
    per_patient_res_preview = {
        pid: int(sum(
            1 for rec in resistance_df.to_dict("records")
            if isinstance(rec.get("genes_mentioned"), list)
            and len(functional_genes_per_patient.get(pid, set())
                    & {g.upper() for g in rec["genes_mentioned"]}) >= MIN_RES_GENE_OVERLAP
        ))
        for pid in sorted(patient_set)
    }
    qc["resistance_per_patient_strict"] = per_patient_res_preview
    max_res = max(per_patient_res_preview.values(), default=0)
    qc["max_resistance_per_patient_strict"] = int(max_res)
    if max_res > 200:
        qc["resistance_warning"] = (
            f"Max strict resistance records per patient is {max_res}. "
            "Consider raising MIN_RES_GENE_OVERLAP."
        )

with open(NB_OUTPUT / "genomic_qc_report.json", "w", encoding="utf-8") as f:
    json.dump(qc, f, indent=2, default=str)

# ---- Dashboard ----
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# Top 20 genes
top = gene_freq.head(20)
axes[0, 0].barh(range(len(top)), top.values, color="steelblue")
axes[0, 0].set_yticks(range(len(top)))
axes[0, 0].set_yticklabels(top.index, fontsize=8)
axes[0, 0].invert_yaxis()
axes[0, 0].set_xlabel("Mutation count")
axes[0, 0].set_title("Top 20 mutated genes")

# Mutations per patient
pc = cohort_mutations.groupby("patient_id").size().sort_values(ascending=False)
colors = ["firebrick" if p in hypermutated else "darkorange" for p in pc.index]
axes[0, 1].bar(range(len(pc)), pc.values, color=colors, width=0.9)
if len(pc) > 30:
    step = 50 if len(pc) <= 500 else 100
    ticks = list(range(0, len(pc), step))
    if (len(pc) - 1) not in ticks:
        ticks.append(len(pc) - 1)
    axes[0, 1].set_xticks(ticks)
    axes[0, 1].set_xticklabels([str(t + 1) for t in ticks], fontsize=8)
    axes[0, 1].set_xlabel(f"Patient Rank (1 to {len(pc)}, sorted by mutation burden)")
else:
    axes[0, 1].set_xticks(range(len(pc)))
    axes[0, 1].set_xticklabels(pc.index, rotation=45, ha="right", fontsize=7)
    axes[0, 1].set_xlabel("Patient ID")
med_val = float(pc.median())
axes[0, 1].axhline(med_val, color="navy", linestyle="--", linewidth=1.2, label=f"Median ({int(med_val)})")
if len(hypermutated) > 0:
    hyper_thresh = med_val * HYPERMUTATION_FACTOR
    axes[0, 1].axhline(hyper_thresh, color="firebrick", linestyle=":", linewidth=1.5, label=f"Hypermutated cutoff (>{int(hyper_thresh)})")
axes[0, 1].legend(loc="upper right", fontsize=8, framealpha=0.9)
axes[0, 1].set_ylabel("Mutation count")
axes[0, 1].set_title(f"Mutations per patient (N={len(pc)}, {len(hypermutated)} hypermutated)")

# CIViC match granularity
if n_tot:
    ml = pd.Series({"variant": n_var, "gene": n_gen})
    axes[0, 2].bar(ml.index, ml.values, color=["seagreen", "firebrick"])
    axes[0, 2].set_title(f"CIViC match level (variant frac={variant_fraction:.2f})")
    axes[0, 2].set_ylabel("Match count")
else:
    axes[0, 2].text(0.5, 0.5, "No CIViC matches", ha="center", va="center")
    axes[0, 2].set_title("CIViC match level")

# Resistance by source
if len(resistance_df):
    rs = resistance_df["source"].value_counts()
    axes[1, 0].bar(rs.index, rs.values, color="mediumpurple")
    axes[1, 0].set_title("Resistance evidence by source")
    axes[1, 0].set_ylabel("Records")
else:
    axes[1, 0].text(0.5, 0.5, "No resistance evidence", ha="center", va="center")
    axes[1, 0].set_title("Resistance evidence by source")

# Sample type distribution
st = cohort_mutations["sample_type"].value_counts()
axes[1, 1].bar(st.index, st.values, color="teal")
axes[1, 1].set_title("Sample type distribution (cohort)")
axes[1, 1].set_ylabel("Mutations")
axes[1, 1].tick_params(axis="x", rotation=30)

# Variant classification distribution
vc = cohort_mutations["Variant_Classification"].value_counts().head(10)
axes[1, 2].barh(range(len(vc)), vc.values, color="slateblue")
axes[1, 2].set_yticks(range(len(vc)))
axes[1, 2].set_yticklabels(vc.index, fontsize=7)
axes[1, 2].invert_yaxis()
axes[1, 2].set_title("Variant classifications")

plt.tight_layout()

try:
    from IPython.display import Image, display as _display
except ImportError:
    Image = lambda *a, **kw: None
    _display = lambda *a, **kw: None
dash_path = NB_OUTPUT / "genomic_qc_dashboard.png"
plt.savefig(dash_path, dpi=180, bbox_inches="tight")
_display(Image(filename=str(dash_path)))
plt.close(fig)
print("Saved:", dash_path)

# Summary table
display(pd.DataFrame([
    {"metric": k, "value": (str(v)[:160] if isinstance(v, dict) else v)}
    for k, v in qc.items()
]))

,metric,value
0,notebook,02_genomic_pipeline
1,timestamp_utc,2026-10-19T13:19:49.075140+00:00
2,cohort_cases,478
3,cohort_patients,478
4,total_mutations_raw,225653
5,cohort_mutations_filtered,149115
6,patient_mutation_counts,"{'TCGA-05-4244': 219, 'TCGA-05-4249': 344, 'TCGA-05-4250': 359, 'TCGA-05-4382': 1734, 'TCGA-05-4384': 130, 'TCGA-05-4390': 591, 'TCGA-05-4395': 278, 'TCGA-05-43"
7,unique_genes,18311
8,variant_class_distribution,"{'Missense_Mutation': 110661, 'Nonsense_Mutation': 9244, ""3'UTR"": 6635, ""5'UTR"": 3992, 'Splice_Site': 3743, 'Frame_Shift_Del': 3365, 'RNA': 3330, 'Intron': 3308"
9,hypermutated_patients,"[TCGA-05-4382, TCGA-05-4396, TCGA-44-7670, TCGA-49-AARE, TCGA-53-7626, TCGA-55-7907, TCGA-55-7994, TCGA-55-8089, TCGA-55-8506, TCGA-55-A490, TCGA-64-5781, TCGA-69-7979, TCGA-78-7155, TCGA-78-8662, TCGA-86-8073, TCGA-86-8358, TCGA-86-A4JF, TCGA-95-7039, TCGA-95-7043, TCGA-L9-A7SV, TCGA-MN-A4N4, TCGA-NJ-A4YQ]"


Saved: VERIONCO_outputs\02\genomic_qc_dashboard.png


## OUTPUTS — per-patient evidence and tabular artifacts

In [12]:
# ============================================================
# OUTPUTS — per-patient evidence (directory) + tabular artifacts
# ============================================================
per_patient_dir = NB_OUTPUT / "genomic_evidence_per_patient"
per_patient_dir.mkdir(parents=True, exist_ok=True)

# Precompute strict resistance matches per patient
def resistance_matches_patient_strict(rec_genes, patient_functional_genes):
    if not isinstance(rec_genes, list):
        return False
    rec_upper = {g.upper() for g in rec_genes}
    return len(patient_functional_genes & rec_upper) >= MIN_RES_GENE_OVERLAP

patient_evidence = {}
patient_summary_rows = []

for pid in sorted(patient_set):
    p_muts  = cohort_mutations[cohort_mutations["patient_id"] == pid]
    p_genes = gene_summary[gene_summary["patient_id"] == pid]
    p_civic_variant = civic_variant_df[civic_variant_df["patient_id"] == pid] \
                      if len(civic_variant_df) else pd.DataFrame()
    p_civic_gene    = civic_gene_df[civic_gene_df["patient_id"] == pid] \
                      if len(civic_gene_df) else pd.DataFrame()

    functional_genes = functional_genes_per_patient.get(pid, set())
    if len(resistance_df):
        p_res = resistance_df[
            resistance_df["genes_mentioned"].apply(
                lambda rg: resistance_matches_patient_strict(rg, functional_genes))
        ]
    else:
        p_res = pd.DataFrame()

    evidence = {
        "patient_id":   pid,
        "n_mutations":  int(len(p_muts)),
        "n_genes":      int(p_genes["Hugo_Symbol"].nunique()) if len(p_genes) else 0,
        "n_functional_genes": int(len(functional_genes)),
        "is_hypermutated": pid in hypermutated,
        "variants":     p_muts[["Hugo_Symbol", "variant_canonical",
                                "Variant_Classification", "sample_type",
                                "tumor_vaf"]]
                        .to_dict(orient="records") if len(p_muts) else [],
        "genes":        p_genes["Hugo_Symbol"].tolist() if len(p_genes) else [],
        "civic_matches_variant_level":
            p_civic_variant.to_dict(orient="records") if len(p_civic_variant) else [],
        "civic_matches_gene_level_WARNING_NOT_CLINICAL_EVIDENCE":
            p_civic_gene.to_dict(orient="records") if len(p_civic_gene) else [],
        "resistance_evidence_strict":
            p_res.to_dict(orient="records") if len(p_res) else [],
    }
    patient_evidence[pid] = evidence

    # Per-patient JSON
    with open(per_patient_dir / f"{pid}.json", "w", encoding="utf-8") as f:
        json.dump(evidence, f, indent=2, default=str)

    patient_summary_rows.append({
        "patient_id":           pid,
        "n_mutations":          evidence["n_mutations"],
        "n_genes":              evidence["n_genes"],
        "n_functional_genes":   evidence["n_functional_genes"],
        "is_hypermutated":      evidence["is_hypermutated"],
        "n_civic_variant":      len(evidence["civic_matches_variant_level"]),
        "n_civic_gene":         len(evidence["civic_matches_gene_level_WARNING_NOT_CLINICAL_EVIDENCE"]),
        "n_resistance_strict":  len(evidence["resistance_evidence_strict"]),
    })

# Monolithic JSON backup
with open(NB_OUTPUT / "genomic_evidence.json", "w", encoding="utf-8") as f:
    json.dump(patient_evidence, f, indent=2, default=str)

# CSVs
cohort_mutations.to_csv(NB_OUTPUT / "patient_mutations.csv", index=False)
gene_summary.to_csv(NB_OUTPUT / "gene_level_summary.csv", index=False)

if len(civic_variant_df):
    civic_variant_df.to_csv(NB_OUTPUT / "civic_matches_VARIANT_LEVEL.csv", index=False)
if len(civic_gene_df):
    civic_gene_df.to_csv(NB_OUTPUT / "civic_matches_GENE_LEVEL_WARNING_NOT_CLINICAL.csv", index=False)
if len(resistance_df):
    resistance_df.to_csv(NB_OUTPUT / "resistance_evidence.csv", index=False)

summary_df = pd.DataFrame(patient_summary_rows)
summary_df.to_csv(NB_OUTPUT / "genomic_patient_summary.csv", index=False)

display(summary_df)
print(f"\n[✓] per-patient evidence written to: {per_patient_dir}")
print(f"[✓] {len(patient_evidence)} patients processed")

,patient_id,n_mutations,n_genes,n_functional_genes,is_hypermutated,n_civic_variant,n_civic_gene,n_resistance_strict
0,TCGA-05-4244,219,213,180,False,18,39,0
1,TCGA-05-4245,0,0,0,False,0,0,0
2,TCGA-05-4249,344,323,279,False,43,241,0
3,TCGA-05-4250,359,345,294,False,18,30,0
4,TCGA-05-4382,1734,1478,1295,True,1,1115,1
5,TCGA-05-4384,130,127,110,False,0,378,0
6,TCGA-05-4390,591,551,469,False,12,96,1
7,TCGA-05-4395,278,271,218,False,12,234,0
8,TCGA-05-4396,1494,1378,1164,True,0,457,0
9,TCGA-05-4397,857,780,667,False,25,364,0



[✓] per-patient evidence written to: VERIONCO_outputs\02\genomic_evidence_per_patient
[✓] 478 patients processed


## DOCUMENTATION — CIViC granularity + residual gaps

In [13]:
# ============================================================
# DOCUMENTATION — CIViC granularity + residual gaps
# ============================================================
civic_limitation = {
    "civic_file_path":              str(civic_path),
    "civic_file_rows":              int(len(civic_df)),
    "civic_file_columns":           list(civic_df.columns),
    "civic_granularity":            civic_granularity,
    "missing_evidence_fields":      sorted(civic_missing),
    "variant_level_fraction":       float(variant_fraction),
    "workarounds_applied": [
        "Variant-level parsing from the CIViC 'name' field "
        "(e.g. 'BRAF V600E' → gene=BRAF, variant=V600E).",
        "Variant-level CIViC matches written to civic_matches_VARIANT_LEVEL.csv.",
        "Gene-level CIViC matches written to a file whose name includes "
        "GENE_LEVEL_WARNING_NOT_CLINICAL.",
        "Resistance retrieval from PubMed MEDLINE (abstracts + MeSH terms), "
        "checkpointed to JSONL.",
        "Resistance retrieval from ClinicalTrials.gov trial descriptions "
        "and eligibility criteria, checkpointed to JSONL.",
        "Resistance association requires >= MIN_RES_GENE_OVERLAP functional "
        "gene overlaps, not a raw gene-set intersection.",
    ],
    "residual_gaps": [
        "CIViC evidence_direction (supports / does not support) unavailable.",
        "CIViC clinical_significance (sensitivity / resistance / etc.) unavailable.",
        "CIViC evidence_level (A/B/C/D) unavailable.",
        "CIViC evidence_statement full text unavailable.",
    ],
    "downstream_requirements": [
        "Notebook 05 (retrieval) must supply evidence_direction and clinical_significance "
        "via direct retrieval (PubMed, CIViC API if internet is enabled, or a full "
        "CIViC nightly evidence-items TSV).",
        "Notebook 06 (planners) must not treat gene-level CIViC matches as "
        "variant-specific clinical evidence.",
        "Notebooks 06/07 must not treat resistance_evidence_strict as "
        "patient-level clinical evidence without further review.",
    ],
    "internet_free_alternative": (
        "If internet is enabled, replace civic_evidence.tsv with the CIViC "
        "nightly clinical evidence summaries TSV, which contains "
        "evidence_direction, clinical_significance, evidence_level, "
        "and evidence_statement."
    ),
    "hypermutation_flag": {
        "factor":   HYPERMUTATION_FACTOR,
        "patients": sorted(hypermutated),
        "note":     "Hypermutated patients dominate gene-level match counts; "
                    "consider excluding them from primary analysis.",
    },
}
with open(NB_OUTPUT / "civic_data_limitation.json", "w", encoding="utf-8") as f:
    json.dump(civic_limitation, f, indent=2)

print("=" * 60)
print("CIViC DATA LIMITATION SUMMARY")
print("=" * 60)
print(f"  Granularity:               {civic_limitation['civic_granularity']}")
print(f"  Missing evidence fields:   {civic_limitation['missing_evidence_fields']}")
print(f"  Variant-level fraction:    {variant_fraction*100:.1f}%")
print(f"  Hypermutated patients:     {sorted(hypermutated)}")
print("=" * 60)

CIViC DATA LIMITATION SUMMARY
  Granularity:               feature_level
  Missing evidence fields:   ['description']
  Variant-level fraction:    4.5%
  Hypermutated patients:     ['TCGA-05-4382', 'TCGA-05-4396', 'TCGA-44-7670', 'TCGA-49-AARE', 'TCGA-53-7626', 'TCGA-55-7907', 'TCGA-55-7994', 'TCGA-55-8089', 'TCGA-55-8506', 'TCGA-55-A490', 'TCGA-64-5781', 'TCGA-69-7979', 'TCGA-78-7155', 'TCGA-78-8662', 'TCGA-86-8073', 'TCGA-86-8358', 'TCGA-86-A4JF', 'TCGA-95-7039', 'TCGA-95-7043', 'TCGA-L9-A7SV', 'TCGA-MN-A4N4', 'TCGA-NJ-A4YQ']


## OUTPUTS — final inventory & completion banner

In [14]:
import gc
# ============================================================
# OUTPUTS — final inventory & completion banner
# ============================================================
import shutil

outputs = []
for p in sorted(NB_OUTPUT.rglob("*")):
    if p.is_file():
        outputs.append({
            "file":       str(p.relative_to(NB_OUTPUT)),
            "size_bytes": p.stat().st_size,
        })
display(pd.DataFrame(outputs))

print("\n" + "=" * 50)
print("VERIONCO NOTEBOOK 02 COMPLETE")
print("=" * 50)
print(f"Artifacts: {NB_OUTPUT}")

zip_target = OUTPUT_ROOT / "02_artifacts"
shutil.make_archive(str(zip_target), "zip", str(NB_OUTPUT))
print(f"Zipped:    {zip_target}.zip")

# Free memory that will not be needed downstream
try:
    del mutation_df
except NameError:
    pass
gc.collect()

,file,size_bytes
0,civic_data_limitation.json,3381
1,civic_matches_GENE_LEVEL_WARNING_NOT_CLINICAL.csv,12880475
2,civic_matches_VARIANT_LEVEL.csv,630984
3,gene_level_summary.csv,8830630
4,genomic_evidence.json,90625393
5,genomic_evidence_per_patient\TCGA-05-4244.json,82749
6,genomic_evidence_per_patient\TCGA-05-4245.json,306
7,genomic_evidence_per_patient\TCGA-05-4249.json,234643
8,genomic_evidence_per_patient\TCGA-05-4250.json,110979
9,genomic_evidence_per_patient\TCGA-05-4382.json,1018909



VERIONCO NOTEBOOK 02 COMPLETE
Artifacts: VERIONCO_outputs\02
Zipped:    VERIONCO_outputs\02_artifacts.zip


## Handoff to Notebook 03 (Clinical Extraction)

**Inputs from Notebook 02:**

- `patient_mutations.csv` — per-patient filtered variants, with `sample_type`
- `gene_level_summary.csv` — per-patient gene aggregation
- `civic_matches_VARIANT_LEVEL.csv` — variant-level CIViC matches only
- `civic_matches_GENE_LEVEL_WARNING_NOT_CLINICAL.csv` — gene-level matches,
  labelled with a warning in the filename
- `genomic_evidence_per_patient/{patient_id}.json` — one file per patient
- `genomic_evidence.json` — monolithic backup
- `resistance_evidence.csv` — resistance signals from PubMed + ClinicalTrials
- `genomic_patient_summary.csv` — quick per-patient overview
- `patient_modality_overlap.csv` — genomics / pathology patient overlap
- `civic_data_limitation.json` — documented CIViC granularity gaps

**Leakage contract:** every genomic evidence item carries `patient_id`.
Notebook 03 joins clinical features on the same key.

**Critical limitations to carry forward:**

1. CIViC matches are gene-level where variant-level evidence is unavailable.
   Downstream notebooks must NOT treat gene-level matches as clinical evidence.
2. `evidence_direction`, `clinical_significance`, `evidence_level`, and
   `evidence_statement` are missing from the supplied CIViC file. Notebook 05
   must retrieve these via direct API access if internet is available.
3. Resistance evidence is associated at the gene-set level, requires
   `MIN_RES_GENE_OVERLAP` functional-gene overlaps, and is not patient-specific
   in the source data. Downstream notebooks must treat it as a retrieval hint,
   not a clinical claim.
4. Hypermutated patients are flagged in `genomic_qc_report.json`. Consider
   excluding them from primary analysis and reporting them separately.

In [15]:
# ============================================================
# EXPORT & DOWNLOAD ARTIFACT BUNDLE
# ============================================================
import shutil
from pathlib import Path
try:
    from IPython.display import FileLink, display
except ImportError:
    def FileLink(p): return p
    def display(*a): pass

export_dir = OUTPUT_ROOT / 'exports'
export_dir.mkdir(parents=True, exist_ok=True)
zip_name = f'verionco_outputs_{NB_OUTPUT.name}'
zip_path = export_dir / zip_name

# Package NB_OUTPUT into zip
shutil.make_archive(str(zip_path), 'zip', NB_OUTPUT)
final_zip = export_dir / f'{zip_name}.zip'

print('=' * 60)
print(f'VERIONCO NOTEBOOK {NB_OUTPUT.name} COMPLETE')
print(f'Artifact bundle created: {final_zip}')
print(f'Archive size: {final_zip.stat().st_size / (1024*1024):.2f} MB')
print('=' * 60)

# Display download link (clickable in Kaggle/Jupyter)
display(FileLink(str(final_zip)))


VERIONCO NOTEBOOK 02 COMPLETE
Artifact bundle created: VERIONCO_outputs\exports\verionco_outputs_02.zip
Archive size: 47.42 MB
